In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
create widget text storageName default "adlsproyectofinaletl";

In [0]:
--Crear container delta-lake previamente en el datalake de la celda 2
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-delta-lake`
URL 'abfss://delta-lake@adlsproyectofinaletl.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential-proyctoetl`)
COMMENT 'Ubicación externa para las tablas delta del Data deltaake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@adlsproyectofinaletl.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential-proyctoetl`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@adlsproyectofinaletl.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential-proyctoetl`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
DROP CATALOG IF EXISTS catalog_proyectoetl CASCADE;

In [0]:
--Crear container metastore previamente en el datalake de la celda 2
CREATE CATALOG IF NOT EXISTS catalog_proyectoetl
MANAGED LOCATION 'abfss://metastore@adlsproyectofinaletl.dfs.core.windows.net/'
COMMENT 'Catalogo para almacenar las tablas metastore;'

In [0]:
%sql
DROP CATALOG IF EXISTS catalog_au CASCADE;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS catalog_au
MANAGED LOCATION 'abfss://metastore@adlsproyectofinaletl.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
%sql
DROP SCHEMA IF EXISTS catalog_au.raw;
DROP SCHEMA IF EXISTS catalog_au.bronze;
DROP SCHEMA IF EXISTS catalog_au.silver;
DROP SCHEMA IF EXISTS catalog_au.golden;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalog_au.raw;
CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_au.silver;
CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.detalle_ventas (
tipo_comprobante string,
nro_comprobante string,
cod_producto string,
precio double,
cantidad int,
subtotal double
)
USING DELTA
LOCATION "abfss://bronze@adlsproyectofinaletl.dfs.core.windows.net/detalle_ventas"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.producto (
descripcion string,
id_producto integer,
precio double
)
USING DELTA
LOCATION "abfss://bronze@adlsproyectofinaletl.dfs.core.windows.net/producto"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.ventas (
  fecha_venta timestamp,
  tipo_comprobante string,
  nro_comprobante string,
  tipo_doc string,
  nro_doc string,
  cliente string,
  precio double,
  dscto double,
  total double
)
USING DELTA
LOCATION "abfss://bronze@adlsproyectofinaletl.dfs.core.windows.net/ventas"

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.silver.ventas_transformed (
  fecha timestamp,
  tipo_compr string,
  nro_compr integer,
  tipo_doc string,
  desc_cliente string,
  cod_prod string,
  precio_prod double, 
  dscto double, 
  total double, 
  ingestion_date timestamp,
  cant_prod int ,
  des_prod string
)
USING DELTA
LOCATION "abfss://silver@adlsproyectofinaletl.dfs.core.windows.net/ventas_transformed"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.golden.golden_ventas_partitioned (
  race_year integer,
  conteo long,
  desc_prod string,
  total_venta  DOUBLE
)
USING DELTA
LOCATION "abfss://golden@adlsproyectofinaletl.dfs.core.windows.net/golden_ventas_partitioned"